# 💬 AskAway — ask questions about your own documents, with sources

**How this notebook works**
1. Run the cells top to bottom (setup → core → app).
2. Use the interactive panel below the *app* cell. **Nothing is pre-filled: every result is generated from what you upload or type.** Each result shows a *Provenance* box (file name, size, SHA-256 fingerprint) so you can prove it came from your input.
3. Failures never crash the notebook: a bad file or API hiccup gives a readable message and you can simply correct the input and press the button again.
4. No widgets (rare)? Run `run_cli()` in the fallback cell for an `input()`-based version.

## 1. Setup

In [ ]:
# One-time setup (about 1-3 minutes in Colab). Safe to re-run.
import subprocess, sys, shutil
def pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    if r.returncode: print("pip warning for", pkgs, "->", r.stderr[-300:])
pip("anthropic", "pypdf", "ipywidgets", "numpy", "scikit-learn", "sentence-transformers", "pytesseract", "pdf2image", "graphviz")
if shutil.which("tesseract") is None or shutil.which("pdftoppm") is None:      # OCR for scanned PDFs
    subprocess.run("apt-get -qq update && apt-get -qq install -y tesseract-ocr poppler-utils graphviz", shell=True, capture_output=True)
print("OCR available:", shutil.which("tesseract") is not None, "| setup done")

## 2. Core utilities

In [ ]:
import os, sys, io, re, json, time, math, hashlib, getpass, subprocess, tempfile, html, traceback, difflib, warnings
from pathlib import Path
from dataclasses import dataclass, field
warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ config
MAX_FILE_MB = 25                                   # reject bigger uploads
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5-5")
STORE_DIR = os.environ.get("STUDYAI_STORE", "studyai_store")   # point this at Google Drive to persist across sessions

class InputError(Exception):
    """A problem with the user's input that the user can fix (bad file, empty, unsupported...)."""

class AIError(Exception):
    """A problem talking to the AI backend (bad key, rate limit, network...)."""

def sha256(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def esc(s) -> str:
    return html.escape(str(s))

# ------------------------------------------------------------------ AI backend (swap this one function to change provider)
_client = None

def _get_api_key():
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        try:
            from google.colab import userdata          # Colab "Secrets" panel
            key = userdata.get("ANTHROPIC_API_KEY")
        except Exception:
            key = None
    if not key:
        key = getpass.getpass("Paste your Anthropic API key (input hidden): ").strip()
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key
    return key

def reset_api_key():
    """Call this if you pasted a wrong key."""
    global _client
    _client = None
    os.environ.pop("ANTHROPIC_API_KEY", None)

def llm(prompt: str, system: str = "", max_tokens: int = 3000, retries: int = 4) -> str:
    """One text-in/text-out call with retry + back-off for rate limits and transient failures."""
    global _client
    import anthropic
    if _client is None:
        key = _get_api_key()
        if not key:
            raise AIError("No API key provided. Run the API-key cell again.")
        _client = anthropic.Anthropic(api_key=key, max_retries=0)
    delay = 3
    last = None
    for attempt in range(retries + 1):
        try:
            r = _client.messages.create(
                model=MODEL, max_tokens=max_tokens,
                system=system or "You are a careful, accurate study assistant.",
                messages=[{"role": "user", "content": prompt}])
            return "".join(b.text for b in r.content if getattr(b, "type", "") == "text")
        except anthropic.AuthenticationError:
            reset_api_key()
            raise AIError("The API key was rejected. Run reset_api_key() and re-enter a valid key.")
        except anthropic.RateLimitError as e:
            last = "rate limit reached"
        except (anthropic.APIConnectionError, anthropic.APITimeoutError) as e:
            last = "network problem"
        except anthropic.APIStatusError as e:
            if e.status_code in (429, 500, 502, 503, 529):
                last = f"temporary server error ({e.status_code})"
            else:
                raise AIError(f"API error {e.status_code}: {getattr(e, 'message', e)}")
        if attempt < retries:
            print(f"   ... {last}; retrying in {delay}s ({attempt + 1}/{retries})")
            time.sleep(delay)
            delay *= 2
    raise AIError(f"AI backend unavailable after {retries + 1} attempts ({last}). Wait a minute and try again.")

def parse_json_loose(text: str):
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    a, b = t.find("{"), t.rfind("}")
    if a != -1 and b > a:
        return json.loads(t[a:b + 1])
    raise ValueError("no JSON object found")

def llm_json(prompt: str, system: str = "", max_tokens: int = 3500) -> dict:
    raw = llm(prompt, system, max_tokens)
    try:
        return parse_json_loose(raw)
    except Exception:
        fixed = llm("The following was supposed to be a single valid JSON object but is malformed. "
                    "Return ONLY the corrected JSON object, nothing else.\n\n" + raw, "You repair JSON.", max_tokens)
        try:
            return parse_json_loose(fixed)
        except Exception as e:
            raise AIError(f"The AI returned a response that could not be parsed ({e}). Try again.")

# ------------------------------------------------------------------ validation
DOC_EXT = {".pdf", ".txt"}
AUDIO_EXT = {".mp3", ".wav", ".m4a"}

def validate_upload(name: str, data: bytes, allowed: set, max_mb: float = None) -> str:
    """Return the lower-case extension, or raise InputError with a fixable message."""
    max_mb = MAX_FILE_MB if max_mb is None else max_mb
    ext = Path(name).suffix.lower()
    if ext not in allowed:
        raise InputError(f"'{name}': unsupported format '{ext or 'none'}'. Allowed: {', '.join(sorted(allowed))}")
    if data is None or len(data) == 0:
        raise InputError(f"'{name}' is empty (0 bytes). Choose a different file.")
    if len(data) > max_mb * 1024 * 1024:
        raise InputError(f"'{name}' is {len(data) / 1048576:.1f} MB; the limit is {max_mb} MB. Split or compress it.")
    if ext == ".pdf" and b"%PDF" not in data[:1024]:
        raise InputError(f"'{name}' does not look like a real PDF (wrong header). It may be a renamed file.")
    if ext == ".wav" and not (data[:4] == b"RIFF" and data[8:12] == b"WAVE"):
        raise InputError(f"'{name}' does not look like a real WAV file (wrong header).")
    return ext

# ------------------------------------------------------------------ small text helpers
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", str(s).lower())).strip()

def words(s: str) -> list:
    return re.findall(r"\w+", str(s).lower())

def jaccard(a: str, b: str) -> float:
    A, B = set(words(a)), set(words(b))
    return len(A & B) / max(1, len(A | B))

def split_text(t: str, size: int = 900, overlap: int = 150) -> list:
    t = t.strip()
    out, i = [], 0
    while i < len(t):
        j = min(len(t), i + size)
        if j < len(t):
            k = max(t.rfind(". ", i + size // 2, j), t.rfind("\n", i + size // 2, j))
            if k != -1:
                j = k + 1
        out.append(t[i:j].strip())
        if j >= len(t):
            break
        i = max(j - overlap, i + 1)
    return [c for c in out if len(c) >= 30 or len(out) == 1]

# ------------------------------------------------------------------ notebook UI helpers
try:
    import ipywidgets as W
    from IPython.display import display, clear_output, HTML, Markdown
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False
    try:
        from IPython.display import display, clear_output, HTML, Markdown
    except Exception:
        display = print
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()     # lets ipywidgets (incl. FileUpload) work in Colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CSS = """<style>
.sn-card{border:1px solid #8884;border-radius:10px;padding:12px 16px;margin:8px 0}
.sn-ok{background:#2e7d3218;border-left:4px solid #2e7d32}.sn-warn{background:#ef6c0018;border-left:4px solid #ef6c00}
.sn-bad{background:#c6282818;border-left:4px solid #c62828}.sn-info{background:#1565c018;border-left:4px solid #1565c0}
.sn-mark{background:#ffeb3b88;padding:0 2px}.sn-unc{background:#ff980055;border-bottom:2px dotted #e65100;padding:0 2px}
.sn-small{font-size:.85em;opacity:.8}.sn-table td,.sn-table th{padding:4px 10px;border-bottom:1px solid #8883;text-align:left;vertical-align:top}
</style>"""

def show_html(s: str):
    display(HTML(CSS + s))

def callout(kind: str, msg: str) -> str:
    icon = {"ok": "✅", "warn": "⚠️", "bad": "⛔", "info": "ℹ️"}[kind]
    return f"<div class='sn-card sn-{kind}'>{icon} {msg}</div>"

def uploaded_items(up) -> list:
    """Normalise ipywidgets FileUpload .value across v7 (dict) and v8 (tuple) -> [(name, bytes)]."""
    v = up.value
    items = []
    if isinstance(v, dict):
        for name, meta in v.items():
            items.append((name, bytes(meta["content"])))
    else:
        for f in v:
            items.append((f["name"], bytes(f["content"])))
    return items

def reset_upload(up):
    for fn in (lambda: up.value.clear(), lambda: setattr(up, "value", ()), lambda: setattr(up, "_counter", 0)):
        try:
            fn()
        except Exception:
            pass

def cli_upload(allowed: set) -> list:
    """Fallback when ipywidgets is unavailable: Colab file picker, else typed paths."""
    if IN_COLAB:
        from google.colab import files
        up = files.upload()
        return [(n, d) for n, d in up.items()]
    raw = input(f"Path(s) to file(s) ({', '.join(sorted(allowed))}), separated by commas: ")
    out = []
    for p in [x.strip() for x in raw.split(",") if x.strip()]:
        try:
            out.append((Path(p).name, Path(p).read_bytes()))
        except Exception as e:
            print(f"⚠️ Cannot read '{p}': {e}")
    return out

def friendly(fn, *a, **k):
    """Run fn; print a readable message instead of crashing the notebook."""
    try:
        return fn(*a, **k)
    except InputError as e:
        show_html(callout("warn", f"<b>Input problem:</b> {esc(e)}<br><span class='sn-small'>Fix the input and try again — nothing else was affected.</span>"))
    except AIError as e:
        show_html(callout("bad", f"<b>AI backend:</b> {esc(e)}<br><span class='sn-small'>Your uploaded data is still loaded; just press the button again.</span>"))
    except Exception as e:
        show_html(callout("bad", f"<b>Unexpected error:</b> {esc(type(e).__name__)}: {esc(e)}<br><span class='sn-small'>The notebook is still usable. Details below.</span>"))
        traceback.print_exc(limit=3)
    return None

In [ ]:
import numpy as np

@dataclass
class LoadedDoc:
    doc_id: str
    name: str
    sha: str
    size: int
    kind: str                      # "pdf" or "txt"
    pages: list                    # [(page_or_section_no, text)]
    student: str = ""
    notes: list = field(default_factory=list)
    ocr_pages: list = field(default_factory=list)

    @property
    def text(self) -> str:
        return "\n\n".join(t for _, t in self.pages)

    @property
    def text_with_pages(self) -> str:
        lab = "Page" if self.kind == "pdf" else "Section"
        return "\n\n".join(f"[{lab} {n}]\n{t}" for n, t in self.pages)

    @property
    def label(self) -> str:
        return "p." if self.kind == "pdf" else "§"

def clean_text(t: str) -> str:
    t = (t or "").replace("\x00", "")
    t = re.sub(r"-\n(\w)", r"\1", t)
    t = re.sub(r"[ \t\f\v]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def _ocr_pages(data: bytes, page_numbers: list):
    try:
        import pytesseract
        from pdf2image import convert_from_bytes
        pytesseract.get_tesseract_version()
    except Exception:
        return {}, "OCR is not installed (run the install cell, which tries to apt-get install tesseract-ocr + poppler-utils)."
    out, err = {}, None
    for n in page_numbers[:80]:
        try:
            img = convert_from_bytes(data, dpi=200, first_page=n, last_page=n)[0]
            out[n] = pytesseract.image_to_string(img)
        except Exception as e:
            err = f"OCR failed on page {n}: {e}"
    return out, err

def extract_pdf_pages(data: bytes, name: str, use_ocr: bool = True):
    try:
        from pypdf import PdfReader
        reader = PdfReader(io.BytesIO(data))
        if reader.is_encrypted:
            try:
                ok = reader.decrypt("")
            except Exception:
                ok = 0
            if not ok:
                raise InputError(f"'{name}' is password-protected. Remove the password and upload again.")
        raw = []
        for i, p in enumerate(reader.pages, 1):
            try:
                raw.append((i, clean_text(p.extract_text() or "")))
            except Exception:
                raw.append((i, ""))
    except InputError:
        raise
    except Exception as e:
        raise InputError(f"'{name}' could not be read as a PDF (corrupt or damaged): {e}")
    if not raw:
        raise InputError(f"'{name}' has no pages.")
    notes, ocr_done = [], []
    blank = [n for n, t in raw if len(t) < 25]
    if blank and use_ocr:
        res, err = _ocr_pages(data, blank)
        d = dict(raw)
        for n, t in res.items():
            t = clean_text(t)
            if len(t) >= 25:
                d[n] = t
                ocr_done.append(n)
        raw = sorted(d.items())
        if ocr_done:
            notes.append(f"OCR was used on {len(ocr_done)} page(s): {ocr_done[:15]}{'...' if len(ocr_done) > 15 else ''} — OCR text may contain errors.")
        if err and len(res) < len(blank):
            notes.append(err)
    still_blank = [n for n, t in raw if len(t) < 25]
    if len(still_blank) == len(raw):
        raise InputError(f"'{name}' looks like a scanned/image-only PDF and no text could be extracted"
                         f"{' (OCR unavailable or found nothing)' if use_ocr else ' (OCR is switched off)'}. "
                         "Enable OCR / install tesseract, or upload a text-based PDF.")
    if still_blank:
        notes.append(f"{len(still_blank)} page(s) had no readable text and were skipped: {still_blank[:15]}")
    pages = [(n, t) for n, t in raw if len(t) >= 25]
    return pages, ocr_done, notes

def decode_text(data: bytes, name: str) -> str:
    for enc in ("utf-8-sig", "utf-16", "cp1252", "latin-1"):
        try:
            t = data.decode(enc)
            if t.count("\x00") > len(t) * 0.01:
                continue
            return t
        except Exception:
            continue
    raise InputError(f"'{name}' could not be decoded as text.")

def load_document(name: str, data: bytes, student: str = "", use_ocr: bool = True) -> LoadedDoc:
    ext = validate_upload(name, data, DOC_EXT)
    sha = sha256(data)
    if ext == ".pdf":
        pages, ocr_done, notes = extract_pdf_pages(data, name, use_ocr)
        kind = "pdf"
    else:
        t = clean_text(decode_text(data, name))
        if len(t) < 25:
            raise InputError(f"'{name}' contains almost no text ({len(t)} characters).")
        parts = split_text(t, 3000, 0) or [t]
        pages, ocr_done, notes, kind = list(enumerate(parts, 1)), [], [], "txt"
    total = sum(len(t) for _, t in pages)
    if total < 100:
        raise InputError(f"'{name}' has too little text ({total} characters) to work with.")
    return LoadedDoc(doc_id=sha[:10], name=name, sha=sha, size=len(data), kind=kind, pages=pages,
                     student=(student or "").strip(), notes=notes, ocr_pages=ocr_done)

# ------------------------------------------------------------------ persistence ("previously uploaded documents")
def save_doc(doc: LoadedDoc):
    try:
        Path(STORE_DIR).mkdir(parents=True, exist_ok=True)
        Path(STORE_DIR, f"{doc.doc_id}.json").write_text(json.dumps(doc.__dict__, ensure_ascii=False))
    except Exception as e:
        print(f"(could not save '{doc.name}' to {STORE_DIR}: {e})")

def load_saved_docs() -> list:
    out = []
    for p in sorted(Path(STORE_DIR).glob("*.json")) if Path(STORE_DIR).exists() else []:
        try:
            d = json.loads(p.read_text())
            d["pages"] = [tuple(x) for x in d["pages"]]
            out.append(LoadedDoc(**d))
        except Exception:
            pass
    return out

# ------------------------------------------------------------------ chunking + retrieval
def chunk_doc(doc: LoadedDoc, size: int = 900, overlap: int = 150) -> list:
    chunks = []
    for page, text in doc.pages:
        for j, c in enumerate(split_text(text, size, overlap)):
            chunks.append(dict(doc_id=doc.doc_id, doc_name=doc.name, student=doc.student,
                               page=page, label=doc.label, text=c))
    return chunks

class Index:
    """Dense retrieval (sentence-transformers) with an automatic TF-IDF fallback."""
    def __init__(self):
        self.chunks, self._E, self._tf, self._M = [], None, None, None
        try:
            from sentence_transformers import SentenceTransformer
            self._st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
            self.backend = "sentence-transformers (dense)"
        except Exception as e:
            self._st, self.backend = None, "TF-IDF (fallback)"
        self.min_relevance = 0.30 if self._st is not None else 0.10     # below this, passages are "not relevant"
        self.support_score = 0.45 if self._st is not None else 0.20

    def has_doc(self, doc_id): return any(c["doc_id"] == doc_id for c in self.chunks)

    def add(self, doc: LoadedDoc):
        if self.has_doc(doc.doc_id):
            return
        new = chunk_doc(doc)
        if self._st is not None and new:
            E = self._st.encode([c["text"] for c in new], normalize_embeddings=True, show_progress_bar=False)
            for c, e in zip(new, E):
                c["emb"] = e
        self.chunks += new
        self._refit()

    def remove(self, doc_id):
        self.chunks = [c for c in self.chunks if c["doc_id"] != doc_id]
        self._refit()

    def _refit(self):
        if not self.chunks:
            self._E = self._M = self._tf = None
            return
        if self._st is not None:
            self._E = np.vstack([c["emb"] for c in self.chunks])
        else:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self._tf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
            self._M = self._tf.fit_transform([c["text"] for c in self.chunks])

    def _scores(self, q):
        if self._st is not None:
            return self._E @ self._st.encode([q], normalize_embeddings=True, show_progress_bar=False)[0]
        return (self._M @ self._tf.transform([q]).T).toarray().ravel()

    def search(self, query, k=5, doc_ids=None, student=None):
        if not self.chunks or not str(query).strip():
            return []
        s = self._scores(query)
        order = np.argsort(-s)
        out = []
        for i in order:
            c = self.chunks[i]
            if doc_ids and c["doc_id"] not in doc_ids:
                continue
            if student and student.strip().lower() not in ("", "all students") and c["student"].lower() != student.strip().lower():
                continue
            r = {k_: v for k_, v in c.items() if k_ != "emb"}
            r["score"] = float(s[i])
            out.append(r)
            if len(out) >= k:
                break
        return out

In [ ]:
# Enter your Anthropic API key once. Order tried: environment variable -> Colab Secrets (ANTHROPIC_API_KEY) -> hidden prompt.
# The AI backend is the single function llm() in the next cell — swap it to use another provider.
_get_api_key()
print("Key loaded. Model:", MODEL)

## 3. AskAway app
1. Upload one or many PDF/TXT files (optionally tag them with a **student** name/ID) → **Add to library**.
2. Choose where to search (all documents or one), an optional student filter, an **answer style** (Simple / Detailed / Exam-oriented / Example-based) and how many passages to retrieve.
3. Type any question → **Ask**. Keep asking follow-ups (ticked by default) or press **New conversation**.

Every answer lists the retrieved passages with document, page and similarity score. If your notes don't cover the question, AskAway says so instead of guessing.

In [ ]:
DOCS = {}                       # doc_id -> LoadedDoc
INDEX = Index()
HISTORY = []                    # [{"q":..., "standalone":..., "a":..., "sufficient":...}]
print(f"Retrieval backend: {INDEX.backend}  (relevance threshold {INDEX.min_relevance})")

STYLES = {
    "Simple": "Explain in plain, short language a beginner would understand (3-5 sentences).",
    "Detailed": "Give a thorough, well-structured explanation covering all relevant details found in the passages.",
    "Exam-oriented": "Write like a model exam answer: a direct definition/statement first, then numbered key points, and finish with a one-line 'Remember:' tip.",
    "Example-based": "Explain mainly through a concrete worked example. If the example is not in the passages, label it clearly as '(illustrative example — not from your notes)'.",
}
INSUFFICIENT = "The uploaded documents do not contain sufficient information to answer this question."

ANSWER_SYSTEM = ("You answer questions strictly from supplied passages of the student's uploaded documents. "
                 "Never use outside knowledge for facts. If the passages do not contain the answer, set sufficient=false. "
                 "Cite passages as [1], [2] etc. matching the passage numbers.")

def rewrite_followup(q: str) -> str:
    if not HISTORY:
        return q
    ctx = "\n".join(f"Q: {h['q']}\nA: {h['a'][:400]}" for h in HISTORY[-3:])
    r = llm("Rewrite the follow-up question as a fully self-contained question using the conversation for context. "
            "If it is already self-contained, return it unchanged. Output only the question.\n\n"
            f"Conversation:\n{ctx}\n\nFollow-up: {q}", "You rewrite questions.", 150).strip()
    return r.strip('"') or q

def ask(question: str, scope_ids, student, style, k, followup=True) -> dict:
    question = question.strip()
    if not question:
        raise InputError("Type a question first.")
    if not DOCS:
        raise InputError("No documents are loaded yet. Upload at least one PDF/TXT first.")
    standalone = rewrite_followup(question) if followup else question
    hits = INDEX.search(standalone, k=k, doc_ids=scope_ids, student=student)
    if not hits:
        raise InputError("No document matches the selected document/student filter.")
    relevant = [h for h in hits if h["score"] >= INDEX.min_relevance]
    result = dict(question=question, standalone=standalone, hits=hits, used=[], sufficient=False, answer="", missing="")
    if not relevant:
        result["answer"], result["missing"] = INSUFFICIENT, "None of the retrieved passages are relevant to the question."
        return result
    ctx = "\n\n".join(f"[{i}] ({h['doc_name']}, {h['label']}{h['page']})\n{h['text']}" for i, h in enumerate(relevant, 1))
    out = llm_json(f"PASSAGES:\n{ctx}\n\nQUESTION: {standalone}\n\nAnswer style: {STYLES[style]}\n\n"
                   'Return JSON: {"sufficient": true|false, "answer": "...", "used_passages": [1,2], "missing": "what is not covered, if anything"}\n'
                   "If sufficient=false, leave answer empty and describe in 'missing' what the notes lack.", ANSWER_SYSTEM, 1500)
    result["hits"] = relevant + [h for h in hits if h not in relevant]
    result["used"] = [int(u) for u in out.get("used_passages", []) if str(u).isdigit()]
    result["missing"] = out.get("missing", "")
    if out.get("sufficient") and str(out.get("answer", "")).strip():
        result["sufficient"], result["answer"] = True, out["answer"].strip()
    else:
        result["answer"] = INSUFFICIENT
    return result

def render_answer(r: dict):
    h = [f"<div class='sn-card sn-info'><b>Q:</b> {esc(r['question'])}"
         + (f"<br><span class='sn-small'>searched as: {esc(r['standalone'])}</span>" if r["standalone"] != r["question"] else "") + "</div>"]
    if r["sufficient"]:
        h.append(f"<div class='sn-card sn-ok'><b>Answer</b><br><div style='white-space:pre-wrap'>{esc(r['answer'])}</div></div>")
    else:
        h.append(callout("warn", f"<b>{INSUFFICIENT}</b><br>{esc(r['missing'])}<br>"
                                 "<span class='sn-small'>Try uploading notes that cover this topic, or widen the document/student filter. "
                                 "The closest passages found are shown below for transparency.</span>"))
    rows = ""
    for i, p in enumerate(r["hits"], 1):
        used = i in r["used"] and r["sufficient"]
        rel = p["score"] >= INDEX.min_relevance
        rows += (f"<tr><td>[{i}]{' ✅' if used else ''}</td><td>{esc(p['doc_name'])}<br><span class='sn-small'>{esc(p['student'] or '—')}</span></td>"
                 f"<td>{p['label']}{p['page']}</td><td>{p['score']:.2f}{'' if rel else ' (low)'}</td>"
                 f"<td style='white-space:pre-wrap'>{esc(p['text'])}</td></tr>")
    h.append("<h4>Retrieved passages &amp; sources</h4><table class='sn-table'><tr><th>#</th><th>Document / student</th><th>Where</th><th>Score</th><th>Passage</th></tr>"
             + rows + "</table><p class='sn-small'>✅ = passage the answer relied on.</p>")
    show_html("".join(h))

def ingest(items, student, use_ocr):
    added = []
    for name, data in items:
        try:
            d = load_document(name, data, student, use_ocr)
            if d.doc_id in DOCS and DOCS[d.doc_id].student != d.student:
                # same file for a different student -> keep both by salting the id
                d.doc_id = d.doc_id + "-" + re.sub(r"\W", "", d.student.lower())[:8]
            if d.doc_id in DOCS:
                show_html(callout("info", f"<code>{esc(name)}</code> is already loaded — skipped duplicate."))
                continue
            DOCS[d.doc_id] = d
            INDEX.add(d); save_doc(d)
            added.append(d)
            for n in d.notes:
                show_html(callout("warn", f"{esc(name)}: {esc(n)}"))
        except InputError as e:
            show_html(callout("warn", f"<b>Skipped:</b> {esc(e)}"))
        except Exception as e:
            show_html(callout("bad", f"<b>Skipped '{esc(name)}':</b> {esc(type(e).__name__)}: {esc(e)}"))
    return added

def library_html():
    if not DOCS:
        return "<p class='sn-small'>No documents loaded.</p>"
    rows = "".join(f"<tr><td>{esc(d.name)}</td><td>{esc(d.student or '—')}</td><td>{len(d.pages)}</td><td>{len(d.text):,}</td><td><code>{d.sha[:8]}</code></td></tr>" for d in DOCS.values())
    return f"<table class='sn-table'><tr><th>Document</th><th>Student</th><th>Pages</th><th>Chars</th><th>SHA</th></tr>{rows}</table><p class='sn-small'>{len(INDEX.chunks)} searchable passages</p>"

if HAS_WIDGETS:
    up = W.FileUpload(accept=".pdf,.txt", multiple=True, description="Upload notes")
    stu_in = W.Text(placeholder="student name/ID for these files (optional)", description="Student:", layout=W.Layout(width="360px"))
    ocr_cb = W.Checkbox(value=True, description="OCR scanned pages", indent=False)
    b_add = W.Button(description="Add to library", button_style="primary", icon="plus")
    b_clear_lib = W.Button(description="Clear library", icon="trash")
    lib_out = W.Output()
    dd_scope = W.Dropdown(options=[("All uploaded documents", None)], description="Search in:", layout=W.Layout(width="480px"))
    dd_stu = W.Dropdown(options=["All students"], description="Student:")
    dd_style = W.Dropdown(options=list(STYLES), value="Simple", description="Answer style:")
    sl_k = W.IntSlider(value=4, min=1, max=10, description="Passages:")
    fu_cb = W.Checkbox(value=True, description="Treat as follow-up (use previous Q&A as context)", indent=False)
    q_in = W.Textarea(placeholder="Type any question about your uploaded documents…", layout=W.Layout(width="95%", height="70px"))
    b_ask = W.Button(description="Ask", button_style="success", icon="question")
    b_reset = W.Button(description="New conversation", icon="refresh")
    out = W.Output()

    def refresh_controls():
        dd_scope.options = [("All uploaded documents", None)] + [(f"{d.name} ({d.student or 'no student'})", d.doc_id) for d in DOCS.values()]
        dd_stu.options = ["All students"] + sorted({d.student for d in DOCS.values() if d.student})
        with lib_out:
            clear_output(); show_html(library_html())

    def on_add(_):
        items = uploaded_items(up)
        with lib_out:
            clear_output()
            if not items:
                show_html(callout("warn", "Choose file(s) with the upload button first."))
                return
            friendly(ingest, items, stu_in.value, ocr_cb.value)
        reset_upload(up); refresh_controls()

    def on_clear(_):
        DOCS.clear(); INDEX.__init__(); HISTORY.clear(); refresh_controls()

    def on_ask(_):
        with out:
            def run():
                scope = {dd_scope.value} if dd_scope.value else None
                r = ask(q_in.value, scope, dd_stu.value, dd_style.value, sl_k.value, fu_cb.value and bool(HISTORY))
                HISTORY.append(dict(q=r["question"], standalone=r["standalone"], a=r["answer"], sufficient=r["sufficient"]))
                render_answer(r)
            friendly(run)

    def on_reset(_):
        HISTORY.clear()
        with out:
            clear_output(); print("Conversation cleared (documents are kept).")

    b_add.on_click(on_add); b_clear_lib.on_click(on_clear); b_ask.on_click(on_ask); b_reset.on_click(on_reset)
    display(W.VBox([
        W.HTML("<h3>💬 AskAway — your documents, your questions</h3>"),
        W.HBox([up, stu_in, ocr_cb]), W.HBox([b_add, b_clear_lib]), lib_out,
        W.HTML("<hr>"), W.HBox([dd_scope, dd_stu]), W.HBox([dd_style, sl_k]), fu_cb, q_in, W.HBox([b_ask, b_reset]), out]))
else:
    print("ipywidgets not available — use run_cli() below.")

def run_cli():
    """Plain input()-based alternative."""
    if not DOCS:
        friendly(ingest, cli_upload(DOC_EXT), input("Student name (optional): "), True)
    styles = list(STYLES)
    while True:
        q = input("\nQuestion (blank to stop): ").strip()
        if not q:
            break
        st = styles[int(input(f"Style {list(enumerate(styles))} [0]: ") or 0)]
        def run():
            r = ask(q, None, "All students", st, 4, bool(HISTORY))
            HISTORY.append(dict(q=r["question"], standalone=r["standalone"], a=r["answer"], sufficient=r["sufficient"]))
            render_answer(r)
        friendly(run)

### Fallback (only if widgets don't show)
`run_cli()` loads files and asks questions with plain `input()`.

In [ ]:
# run_cli()

## Custom Input Testing and Validation

This section lets you verify, with your own inputs, that the notebook behaves as required. The automated cell tests the *input guards* using files created on the fly; the manual table lists the behaviours to check with your real files.

### Manual acceptance tests
| # | Do this | You should see |
|---|---|---|
| 1 | Upload a notes PDF, ask a question it covers | An answer with `[n]` citations, and the retrieved passages (document, page, score) |
| 2 | Ask a **follow-up** ("explain that more simply") | The question is rewritten using the previous Q&A ("searched as: …") |
| 3 | Switch **Answer style** and ask the same question | Same facts, different presentation (simple / detailed / exam / example) |
| 4 | Ask something **unrelated** to your notes (e.g. "Who won the 2010 World Cup?") | ⚠️ *"The uploaded documents do not contain sufficient information…"* with the closest passages marked low relevance |
| 5 | Upload 2+ documents (give them different **Student** tags) and ask the same question with *Search in* = one document, then a different **Student** filter | Retrieved passages come only from the selected document / student |
| 6 | Upload an empty file, `.docx`, corrupt or scanned PDF | A clear warning; the library and chat keep working |

In [ ]:
def run_validation_tests():
    """Checks the input validators with files built on the fly (these test the *guards*, not the outputs)."""
    from pypdf import PdfWriter
    results = []
    def expect_error(label, fn):
        try:
            fn()
            results.append((label, "❌ accepted (should have been rejected)"))
        except InputError as e:
            results.append((label, f"✅ rejected: {str(e)[:110]}"))
        except Exception as e:
            results.append((label, f"❌ crashed with {type(e).__name__}: {e}"))
    blank = io.BytesIO(); w = PdfWriter(); w.add_blank_page(300, 300); w.write(blank)
    expect_error("empty .txt file", lambda: load_document("empty.txt", b""))
    expect_error("whitespace-only .txt", lambda: load_document("space.txt", b"   \n  \n"))
    expect_error("empty .pdf file", lambda: load_document("empty.pdf", b""))
    expect_error("unsupported format (.docx)", lambda: load_document("notes.docx", b"PK\x03\x04 something"))
    expect_error("renamed/corrupt PDF", lambda: load_document("fake.pdf", b"this is not a pdf at all" * 20))
    expect_error("truncated PDF", lambda: load_document("cut.pdf", b"%PDF-1.4\n1 0 obj\n<<"))
    expect_error("oversized file (limit forced to 0.001 MB)", lambda: validate_upload("big.txt", b"x" * 5000, DOC_EXT, max_mb=0.001))
    expect_error("image-only / scanned-style PDF (blank page, OCR on)", lambda: load_document("scan.pdf", blank.getvalue(), use_ocr=True))
    try:
        d = load_document("ok.txt", ("Photosynthesis converts light energy into chemical energy. " * 6).encode())
        results.append(("valid .txt", f"✅ accepted ({len(d.text)} chars)"))
    except Exception as e:
        results.append(("valid .txt", f"❌ {e}"))
    rows = "".join(f"<tr><td>{esc(a)}</td><td>{esc(b)}</td></tr>" for a, b in results)
    show_html(f"<table class='sn-table'><tr><th>Test</th><th>Result</th></tr>{rows}</table>")
    return all(r[1].startswith("✅") for r in results)



run_validation_tests()

**Filter check** — after loading 2+ documents/students, this cell asks the retrieval layer the same query under each filter and shows that results respect them:

In [ ]:
def filter_check(query):
    if len(DOCS) < 2:
        print("Load at least two documents first."); return
    rows = ""
    for label, kw in [("all documents", {})] + [(f"only '{d.name}'", dict(doc_ids={d.doc_id})) for d in DOCS.values()] + \
                     [(f"student '{s}'", dict(student=s)) for s in sorted({d.student for d in DOCS.values() if d.student})]:
        hits = INDEX.search(query, k=5, **kw)
        rows += f"<tr><td>{esc(label)}</td><td>{esc(sorted({(h['doc_name'], h['student']) for h in hits}))}</td></tr>"
    show_html(f"<table class='sn-table'><tr><th>Filter</th><th>(document, student) pairs returned</th></tr>{rows}</table>")
filter_check(input("Type any query to test the filters: ") if len(DOCS) >= 2 else "")